## Deep Research

One of the classic cross-business Agentic use cases! This is huge.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Commercial implications</h2>
            <span style="color:#00bfff;">A Deep Research agent is broadly applicable to any business area, and to your own day-to-day activities. You can make use of this yourself!
            </span>
        </td>
    </tr>
</table>

In [1]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents.model_settings import ModelSettings
from pydantic import BaseModel, Field
from dotenv import load_dotenv
import asyncio
from IPython.display import display, Markdown
from messenger import send_email, push

In [2]:
load_dotenv(override=True)

True

In [7]:
# Constants 

MODEL_NAME = "gpt-4o-mini"
USE_EMAIL = True
HOW_MANY_SEARCHES = 5

## Strategy for the Deep Research Agent

We are going to do it the bulletproof way. We will build this by code so that we can control step by step.

We are going to orchestrate with code: separate calls to `Runner.run()` for each step in the process.

We will use Structured Outputs at each point.

## We will build 4 Agents:

1. The Search Agent: searches the web for information
2. The Planner Agent: given a question, comes up with a list of searches that should be made
3. The Writer Agent: writes a robust report
4. The Emailer Agent: crafts and sends an email

And then 4 python functions, 1 to call Runner.run() for each of the 4 agents.


## Agent 1: The Search Agent

### OpenAI Hosted Tools

https://openai.github.io/openai-agents-python/tools/#hosted-tools

A paid, quick approach to carrying out managed functionality on OpenAI's cloud.

Their docs surface these tools, but it's worth keeping in mind that they're costly and lock you in to the OpenAI ecosystem. You will need to use OpenAI and OpenAI models; you cannot switch to Gemini or other models, etc. 

If you want more flexibility, use open-source tools or write yourself. 

OpenAI offers the following hosted tools:

`WebSearchTool` lets an agent search the web.  Cost: 1 cent/web search. Not cheap at all. Can be a quick way to prototype when testing out a product. If building for production, should build your own tool.

`FileSearchTool` allows retrieving information from your OpenAI Vector Stores. A fast way to try something out. If you build your own RAG, you would not trust OpenAI to take care of it and then pay extra money for it. 

`CodeInterpreterTool` lets the LLM execute code in a sandboxed environment.  

`HostedMCPTool` exposes a remote MCP server's tools to the model.  It lets you run MCP Servers on OpenAI's hardware, but it's so easy to run them on your own, so this does not seem like a great idea. 

`ImageGenerationTool` generates images from a prompt.  

`ToolSearchTool` lets the model load deferred tools, namespaces, or hosted MCP servers on demand. A tool to search tools, making you more context-efficient in using OpenAI tools.

### Important note - API charge of WebSearchTool

This currently costs 1 cent per call for OpenAI WebSearchTool. That can add up to about $1 for the next 2 labs. We'll use free and low cost Search tools with other platforms, so feel free to skip running this if the cost is a concern. Also student Christian W. pointed out that OpenAI can sometimes charge for multiple searches for a single call, so it could sometimes cost more than 1 cent per call.

Costs are in the Tools section here: https://developers.openai.com/api/docs/pricing


In [27]:
INSTRUCTIONS = """
You are a research assistant. Given a search term, you search the web for that term and 
produce a concise summary of the results. The summary must 2-3 paragraphs and less than 300 words.
Capture the main points and be succinct. Reply only with the summary.
"""

settings = ModelSettings(tool_choice="required")
tools = [WebSearchTool()]

In [28]:
search_agent = Agent(name="Search Agent", instructions=INSTRUCTIONS, tools=tools, model=MODEL_NAME, model_settings=settings)

In [29]:
# Test to see how the search agent works with a specific task example 
task = "Most popular AI Agent frameworks in 2026"

result = await Runner.run(search_agent, task)
display(Markdown(result.final_output))

As of July 2026, several AI agent frameworks have emerged as leaders in the field, each catering to specific development needs:

- **LangGraph**: Recognized for its robust support for complex, stateful workflows, LangGraph is ideal for developers requiring durable execution and streaming capabilities. ([alicelabs.ai](https://alicelabs.ai/en/insights/best-ai-agent-frameworks-2026?utm_source=openai))

- **CrewAI**: Esteemed for its user-friendly approach to role-based multi-agent orchestration, CrewAI facilitates the rapid development of multi-agent systems, making it a top choice for those focusing on agent team coordination. ([alicelabs.ai](https://alicelabs.ai/en/insights/best-ai-agent-frameworks-2026?utm_source=openai))

- **OpenAI Agents SDK**: Tailored for seamless integration with OpenAI's models, this SDK offers tools for agent development, including session management and tracing, catering to developers invested in OpenAI's ecosystem. ([aiunpacking.com](https://aiunpacking.com/blog/top-ai-agent-frameworks-2026/?utm_source=openai))

- **Microsoft Agent Framework**: Combining elements from Semantic Kernel and AutoGen, Microsoft's framework is designed for enterprise applications, particularly within the .NET and Python environments. ([alicelabs.ai](https://alicelabs.ai/en/insights/best-ai-agent-frameworks-2026?utm_source=openai))

- **LlamaIndex**: Specializing in retrieval-augmented generation (RAG) agents, LlamaIndex is optimal for applications centered around document parsing, indexing, and knowledge workflows. ([aiunpacking.com](https://aiunpacking.com/blog/top-ai-agent-frameworks-2026/?utm_source=openai))

These frameworks represent the forefront of AI agent development in 2026, each offering unique features tailored to diverse application requirements. 

### As always, take a look at the trace

https://platform.openai.com/traces

## Agent 2: The Planner Agent

### We will now use Structured Outputs, and include a description of the fields

In [ ]:
# Here, you will define two objectcs. 
# One nuance in using Structured Output is that it's best practice to include reasoning and rationales in the Structured Output. It helps put the model in the reasoning mode when it generates tokens so that the final result will be consistent with the thought process. 
# That will give better output and better performance. 
# That's why it's also important to do the other way around: put the reason first, then the query. So that it thinks through the approach it wants to do then gives a query based on that. 
# If you do the other way around, the model will just give you a query and then just simply give a rationale to justify the query.

# Here, you don't generate object. You actually generate JSON which the agent will turn into object. 
class WebSearchItem(BaseModel):
    reason: str = Field(description="Your reasoning for why this search is important to the query.")
    query: str = Field(description="The search term to use for the web search.")


class WebSearchPlan(BaseModel): # basically a list of WebSearchItem
    searches: list[WebSearchItem] = Field(description="A list of web searches to perform to best answer the query.")

In [14]:
WebSearchPlan.model_json_schema()

{'$defs': {'WebSearchItem': {'properties': {'reason': {'description': 'Your reasoning for why this search is important to the query.',
     'title': 'Reason',
     'type': 'string'},
    'query': {'description': 'The search term to use for the web search.',
     'title': 'Query',
     'type': 'string'}},
   'required': ['reason', 'query'],
   'title': 'WebSearchItem',
   'type': 'object'}},
 'properties': {'searches': {'description': 'A list of web searches to perform to best answer the query.',
   'items': {'$ref': '#/$defs/WebSearchItem'},
   'title': 'Searches',
   'type': 'array'}},
 'required': ['searches'],
 'title': 'WebSearchPlan',
 'type': 'object'}

In [15]:
# See note above about cost of WebSearchTool

INSTRUCTIONS = f"""
You are a research assistant. Given a user query, come up with a set of web searches
to perform to best answer the query. Output {HOW_MANY_SEARCHES} terms to query for.
"""

planner_agent = Agent(name="Planner Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=WebSearchPlan)

In [ ]:
# Test the planner agent with a specific task example (the task is the example from above)
result = await Runner.run(planner_agent, task)
result.final_output

WebSearchPlan(searches=[WebSearchItem(reason='To find up-to-date information about the most popular AI agent frameworks expected in 2026.', query='popular AI agent frameworks 2026'), WebSearchItem(reason='To analyze industry trends and expert predictions regarding AI frameworks in the near future.', query='AI frameworks trends 2026'), WebSearchItem(reason='To gather insights from authoritative tech sources on emerging AI frameworks for agents.', query='top AI agent technologies 2026'), WebSearchItem(reason='To explore reviews and comparisons of AI agent frameworks predicted to gain popularity.', query='best AI agent frameworks comparison 2026'), WebSearchItem(reason='To find discussions or forecasts on AI agent frameworks in tech forums and publications.', query='future of AI agent frameworks 2026')])

## Agent 3: The Writer Agent

In [17]:
INSTRUCTIONS = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name="Writer Agent", instructions=INSTRUCTIONS, model=MODEL_NAME, output_type=ReportData)

In [22]:
# vars(writer_agent)

In [ ]:
# Testing to see what the report looks like. The original code from the instructor did not include this. He trusted the output to put it in Agent 4.
result = await Runner.run(writer_agent, task)
result.final_output


ReportData(short_summary="In 2026, popular AI agent frameworks prominently include OpenAI's GPT series, Google's BERT and T5, Facebook's BlenderBot, and DeepMind's AlphaFold, among others. These frameworks exemplify advancements in natural language processing, conversational agents, and specialized applications like bioinformatics.", markdown_report='# Most Popular AI Agent Frameworks in 2026\n\n## Introduction\nThe landscape of artificial intelligence (AI) frameworks has evolved dramatically, with substantial advances in natural language processing (NLP), computer vision, and multi-modal learning. This report aims to explore the most popular AI agent frameworks as of 2026, highlighting their capabilities, applications, and the underlying technologies that enable their success.\n\n## Overview of AI Agent Frameworks\nAI agents are systems that can interpret data, make decisions, and perform tasks autonomously or semi-autonomously. An effective AI framework typically comprises various co

In [53]:
# vars(ReportData)

## Agent 4: The email agent

In [42]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str) -> str:
    """
    Send out an email with the given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email as plain text
        html_body: The HTML body of the email
    """
    # if USE_EMAIL:
    #     send_email(subject, text_body, html_body)
    # else:
    #     push(f"Subject: {subject}\n\n{text_body}")
    print (f"Subject: {subject}\n\n{text_body}") # Not using email sender or push notifications, so just pritning out the email

In [43]:
send_email_tool.params_json_schema

{'properties': {'subject': {'description': 'The subject of the email',
   'title': 'Subject',
   'type': 'string'},
  'text_body': {'description': 'The body of the email as plain text',
   'title': 'Text Body',
   'type': 'string'},
  'html_body': {'description': 'The HTML body of the email',
   'title': 'Html Body',
   'type': 'string'}},
 'required': ['subject', 'text_body', 'html_body'],
 'title': 'send_email_tool_args',
 'type': 'object',
 'additionalProperties': False}

In [44]:
INSTRUCTIONS = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_agent = Agent(name="Email Agent", instructions=INSTRUCTIONS, tools=[send_email_tool], model=MODEL_NAME)

## Now to Orchestrate by Code

The next 2 functions will plan and execute the search, using the Agents, with calls to `Runner.run()`

We will write 4 functions, each to call a Runner.run()

In [45]:
# Two functions to plan and execute the search

async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.searches
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results


async def search(item: WebSearchItem):
    input_message = f"Search term: {item.query}\nReason for searching: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

The next 2 functions write a report and email it

In [46]:
async def write_report(query: str, search_results: list[str]):
    print("Thinking about report...")
    input_message = f"Original query: {query}\nSummarized search results: {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

### Showtime!

In [47]:
query ="Most popular AI Agent frameworks in 2026"

with trace("Research trace"):
    print("Starting research...")
    search_results = await run_searches(query)
    report = await write_report(query, search_results)
    await send_report_email(report)  
    print("Hooray!")

Starting research...
Planning searches...
Will perform 5 searches
Finished searching
Thinking about report...
Finished writing report
Writing email...
Subject: AI Agent Frameworks in 2026: A Comprehensive Overview

Dear [Recipient],

I hope this email finds you well. 

I’m excited to share a comprehensive overview of AI agent frameworks as we approach the end of 2026. This report encapsulates the transformative landscape of AI agent frameworks, highlighting key players, adoption metrics, and future trends that are reshaping the way organizations implement AI in their operations.

---

## AI Agent Frameworks in 2026: A Comprehensive Overview

### Introduction  
As we delve into 2026, the realm of AI agent frameworks has seen substantial growth and diversification. These frameworks have transitioned from being experimental tools to becoming integral components of core business applications across industries. This report provides a detailed analysis of the most popular AI agent frameworks

### As always, take a look at the trace

https://platform.openai.com/traces

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thanks.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00cc00;">Congratulations on your progress, and a request</h2>
            <span style="color:#00cc00;">You've reached an important moment with the course; you've created a valuable Agent using one of the latest Agent frameworks. You've upskilled, and unlocked new commercial possibilities. Take a moment to celebrate your success!<br/><br/>Something I should ask you -- my editor would smack me if I didn't mention this. If you're able to rate the course on Udemy, I'd be seriously grateful: it's the most important way that Udemy decides whether to show the course to others and it makes a massive difference.<br/><br/>And another reminder to <a href="https://www.linkedin.com/in/eddonner/">connect with me on LinkedIn</a> if you wish! If you wanted to post about your progress on the course, please tag me and I'll weigh in to increase your exposure.
            </span>
        </td>
    </tr>